# 笔记本 A

用来验证两个笔记本是否互相影响。每一格都刻意依赖**前一格**留下的变量，
所以只要串了或断了，后面立刻会报错。

对照文件是 `notebook-B.ipynb`：两边用**同名**的变量（`marker`、`shared`、
`count`）和**同名**的模块级状态，值却不同。

In [9]:
# A 的身份标记。B 里有一个同名但值不同的 marker。
marker = 'AAA'
shared = {'owner': 'A', 'items': [1, 2, 3]}
count = 111
marker, shared, count

('AAA', {'owner': 'A', 'items': [1, 2, 3]}, 111)

In [10]:
# 读回上一格的东西：如果内核被换掉或串到了 B，这里会 NameError 或值不对。
assert marker == 'AAA', f'marker 被改成了 {marker!r}'
assert shared['owner'] == 'A', f'shared 属于 {shared["owner"]!r}'
assert count == 111, f'count 变成了 {count}'
'A 的命名空间完好'

'A 的命名空间完好'

In [11]:
# 原地改动一个可变对象。若两个笔记本共享内核，B 会看到这个改动。
shared['items'].append(999)
shared

{'owner': 'A', 'items': [1, 2, 3, 999]}

In [12]:
# 模块级状态：比变量更隐蔽，也更容易串味。
# 必须显式登记进 sys.modules，否则下一格的 import 会去找磁盘文件而失败。
import sys
import types
sys.modules.pop('probe_shared_state', None)
module = types.ModuleType('probe_shared_state')
module.owner = 'A'
module.value = 111
sys.modules['probe_shared_state'] = module
import probe_shared_state
(probe_shared_state.owner, probe_shared_state.value)

('A', 111)

In [13]:
# 同一个内核里再读一次，确认模块状态属于 A。
import probe_shared_state
assert probe_shared_state.owner == 'A', f'模块被 {probe_shared_state.owner} 占了'
assert probe_shared_state.value == 111
'A 的模块状态完好'

'A 的模块状态完好'

In [14]:
# 工作目录必须是笔记本自己所在的目录，而不是另一个笔记本的。
import os
here = os.getcwd()
sorted(name for name in os.listdir('.') if name.endswith('.ipynb'))

['notebook-A.ipynb', 'notebook-B.ipynb']

In [ ]:
# 写一个只有 A 知道的临时文件，B 里有一格会去读它。
# 用系统临时目录而不是笔记本所在目录，免得把测试文件落进仓库。
# 只回报文件名而不是整条路径：这一格跑完就会把输出写进文件，而整条路径里
# 唯一有意义的事实是文件名 —— 路径只会把某个人的私有目录带进仓库。
import tempfile
from pathlib import Path
probe_file = Path(tempfile.gettempdir()) / 'dsh_probe_a_only.txt'
probe_file.write_text('written by A', encoding='utf-8')
probe_file.name

In [16]:
# 跑它然后按中断：只应打断 A，B 的内核必须不受影响。
import time
for _ in range(120):
    time.sleep(0.5)
print('不会执行到这里')

KeyboardInterrupt: 

In [17]:
# 中断之后 A 自己的内核应该仍然可用，且变量还在。
marker, count

('AAA', 111)

In [18]:
# 关键一格：A 绝不能看见 B 的变量。预期是 NameError。
# 如果它返回了 'BBB'，说明两个笔记本共用了同一个内核。
marker_from_b

NameError: name 'marker_from_b' is not defined